# Lab 4: Location Encoding with Space2Vec and Hyperparameter Tuning (Optuna)

Encodes coordinates with a multi scale Space2Vec style location encoder and combines the
embedding with tabular features in a neural network, then tunes the encoder and network
with Optuna.

Dataset: the California housing point shapefile from Lab 1 (features plus location,
predicting median value). Swap in another point dataset by changing the loading cell.

The notebook uses a compact built in implementation of the Space2Vec multi scale
sinusoidal ("theory") encoder (Mai et al., 2020), so it runs without extra repositories.

In [ ]:
!pip -q install optuna geopandas

In [ ]:
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import torch.nn as nn
import optuna
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_PATH = "california_housing.shp"

## Location encoder

In [ ]:
class TheoryLocationEncoder(nn.Module):
    """Space2Vec "theory" encoder: projects 2D coordinates onto three unit vectors 120 degrees apart,
    applies sin and cos at geometrically spaced scales between min_radius and max_radius,
    then a feed forward layer maps the result to spa_embed_dim."""
    def __init__(self, spa_embed_dim=64, frequency_num=16, max_radius=1.0, min_radius=1e-3, dropout=0.1):
        super().__init__()
        angles = torch.tensor([0.0, 2 * math.pi / 3, 4 * math.pi / 3])
        self.register_buffer("unit_vecs", torch.stack([torch.cos(angles), torch.sin(angles)], dim=1))  # (3, 2)
        log_timescale = math.log(max_radius / min_radius) / max(frequency_num - 1, 1)
        timescales = min_radius * torch.exp(torch.arange(frequency_num).float() * log_timescale)
        self.register_buffer("freqs", 1.0 / timescales)                                                # (F,)
        self.ffn = nn.Sequential(nn.Linear(6 * frequency_num, spa_embed_dim), nn.ReLU(), nn.Dropout(dropout))

    def forward(self, coords):                                   # coords: (batch, 2), roughly in [-1, 1]
        proj = coords @ self.unit_vecs.T                         # (batch, 3)
        scaled = proj.unsqueeze(-1) * self.freqs                 # (batch, 3, F)
        enc = torch.cat([torch.sin(scaled), torch.cos(scaled)], dim=-1).flatten(1)   # (batch, 6F)
        return self.ffn(enc)


class LocationAwareNet(nn.Module):
    def __init__(self, n_features, spa_embed_dim, frequency_num, min_radius, hidden_dim, num_hidden_layer, dropout):
        super().__init__()
        self.loc_enc = TheoryLocationEncoder(spa_embed_dim, frequency_num, 1.0, min_radius, dropout)
        layers, d = [], n_features + spa_embed_dim               # input size includes the location embedding
        for _ in range(num_hidden_layer):
            layers += [nn.Linear(d, hidden_dim), nn.ReLU(), nn.Dropout(dropout)]
            d = hidden_dim
        layers.append(nn.Linear(d, 1))
        self.head = nn.Sequential(*layers)

    def forward(self, feats, coords):
        return self.head(torch.cat([feats, self.loc_enc(coords)], dim=1))

## Data

In [ ]:
gdf = gpd.read_file(DATA_PATH)
gdf = gdf[(gdf["house_age"] < 52) & (gdf["median_val"] < 500001)].copy()
gdf = pd.get_dummies(gdf, columns=["ocean_prox"], dtype="float32")

coords = np.column_stack([gdf.geometry.x, gdf.geometry.y]).astype("float32")
# Normalise coordinates to [-1, 1] (the encoder's extent)
cmin, cmax = coords.min(axis=0), coords.max(axis=0)
coords = 2 * (coords - cmin) / (cmax - cmin) - 1

feats = pd.DataFrame(gdf.drop(columns=["geometry", "median_val"])).astype("float32").values
target = (gdf["median_val"].values / 100_000).astype("float32").reshape(-1, 1)

idx_train, idx_test = train_test_split(np.arange(len(gdf)), test_size=0.2, random_state=SEED)
idx_train, idx_val = train_test_split(idx_train, test_size=0.2, random_state=SEED)

scaler = StandardScaler().fit(feats[idx_train])
feats = scaler.transform(feats).astype("float32")

def tensors(idx):
    return (torch.tensor(feats[idx]).to(device), torch.tensor(coords[idx]).to(device),
            torch.tensor(target[idx]).to(device))

train_t, val_t, test_t = tensors(idx_train), tensors(idx_val), tensors(idx_test)
print(len(idx_train), len(idx_val), len(idx_test))

## Train and evaluate helpers

In [ ]:
def train_and_score(params, train_t, eval_t, epochs=60, batch_size=256):
    model = LocationAwareNet(n_features=train_t[0].shape[1], **{k: params[k] for k in
             ["spa_embed_dim", "frequency_num", "min_radius", "hidden_dim", "num_hidden_layer", "dropout"]}).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=params["lr"])
    loss_fn = nn.MSELoss()
    n = train_t[0].shape[0]
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch_size):
            b = perm[i:i + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(train_t[0][b], train_t[1][b]), train_t[2][b])
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        rmse = torch.sqrt(loss_fn(model(eval_t[0], eval_t[1]), eval_t[2])).item() * 100_000
    return rmse, model

## Optuna search

In [ ]:
def objective(trial):
    params = {
        "spa_embed_dim": trial.suggest_categorical("spa_embed_dim", [16, 32, 64]),
        "frequency_num": trial.suggest_categorical("frequency_num", [16, 32]),
        "min_radius": trial.suggest_float("min_radius", 1e-4, 1e-1, log=True),   # must stay below max_radius = 1
        "num_hidden_layer": trial.suggest_int("num_hidden_layer", 1, 3),
        "hidden_dim": trial.suggest_categorical("hidden_dim", [64, 128, 256]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),
        "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
    }
    val_rmse, _ = train_and_score(params, train_t, val_t)
    return val_rmse

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=30)
print("Best validation RMSE: $", round(study.best_value))
print("Best parameters:", study.best_params)

## Final test score with the best parameters

In [ ]:
test_rmse, best_model = train_and_score(study.best_params, train_t, test_t, epochs=100)
print(f"Test RMSE with tuned location encoder: ${test_rmse:,.0f}")